In [ ]:
# =============================================================================
# FIFE Daily Rainfall (ORNL DAAC ds #29, doi:10.3334/ORNLDAAC/29)
# ALL-IN-ONE: parsing + data-description figures + gauge-density design analysis.
# Paste into ONE Google Colab cell -> Run -> a "Choose Files" button appears ->
# pick the .rar -> it processes everything and downloads fife_outputs.zip.
# =============================================================================
import os, re, sys, glob, subprocess, itertools, warnings
warnings.filterwarnings('ignore')

WORKDIR = 'fife_work'
OUTDIR  = 'fife_full_outputs'
IN_COLAB = 'google.colab' in sys.modules
for d in (WORKDIR, OUTDIR):
    os.makedirs(d, exist_ok=True)

# FIS site grid definition (ORNL DAAC FIFE campaign document):
# SITEGRID_ID = "SSEE-III"; SS = nodes SOUTH of grid origin, EE = nodes EAST.
# 100 x 100 node grid of 200 m cells; origin (NW corner) UTM Z14 N 4,334,000 / E 705,000.
GRID_N0, GRID_E0, CELL = 4_334_000.0, 705_000.0, 200.0
MISSING = -9.9
WET_MM  = 0.5

def _pip(pkg):
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)
    if r.returncode:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                        '--break-system-packages', pkg], check=False)

# ----------------------------- GET THE .RAR FROM YOUR COMPUTER ---------------
def _find_rar():
    hits = sorted(glob.glob('*.rar') + glob.glob('*.RAR'))
    return hits[0] if hits else None

RAR_NAME = _find_rar()
if RAR_NAME is None:
    if IN_COLAB:
        from google.colab import files
        print('>> Pick your .rar file (e.g. fife_sur_met_rain_day_29_1-...rar)')
        up = files.upload()                 # opens the "Choose Files" dialog
        RAR_NAME = [k for k in up if k.lower().endswith('.rar')][0]
    else:
        raise FileNotFoundError('No .rar found in the current folder. '
                                'Put the FIFE .rar next to this script and re-run.')
print('Using archive: %s (%d bytes)' % (RAR_NAME, os.path.getsize(RAR_NAME)))

# ----------------------------- DEPENDENCIES ----------------------------------
try:
    import libarchive
except ImportError:
    _pip('libarchive-c'); import libarchive
try:
    import pyproj
except ImportError:
    _pip('pyproj'); import pyproj
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi': 130, 'savefig.dpi': 300, 'font.size': 9,
                     'axes.grid': True, 'grid.alpha': .3, 'savefig.bbox': 'tight'})

# ----------------------------- EXTRACT GRANULES ------------------------------
# libarchive chokes on the RAR5 directory entry -> extract file entries only.
n_files = 0
with libarchive.file_reader(RAR_NAME) as arch:
    for e in arch:
        if e.isdir or e.size == 0:
            continue
        dst = os.path.join(WORKDIR, os.path.basename(e.pathname))
        with open(dst, 'wb') as fh:
            for blk in e.get_blocks():
                fh.write(blk)
        n_files += 1
print('Extracted %d granule files -> %s/' % (n_files, WORKDIR))

# ----------------------------- 3. PARSE .RDY GRANULES ------------------------
# Granule layout: record 1 = header, records 2-4 = FIS cross-links,
# record 5 = column names, records 6+ = data.  Line endings are CR/LF and the
# files are DOS-terminated with 0x1A, so split on any CR/LF run.
COLS = ['sitegrid', 'station', 'date', 'obstype', 'rain',
        'comments', 'cert', 'revdate']
rows = []
for f in sorted(glob.glob(os.path.join(WORKDIR, '*.rdy')) +
                glob.glob(os.path.join(WORKDIR, '*.RDY'))):
    txt = open(f, encoding='latin-1').read().replace('\x1a', '')
    lines = [l.strip() for l in re.split(r'[\r\n]+', txt) if l.strip()]
    hdr = [i for i, l in enumerate(lines) if l.upper().startswith('SITEGRID_ID')]
    if not hdr:
        print('  !! no column record in', os.path.basename(f)); continue
    for l in lines[hdr[0] + 1:]:
        p = [x.strip().strip("'").strip() for x in l.split(',')]
        if len(p) >= 8:
            rows.append(p[:8] + [os.path.basename(f)])
df = pd.DataFrame(rows, columns=COLS + ['granule'])
print('Parsed %d observation records from %d granules' % (len(df), n_files))

# ----------------------------- 4. CLEAN / TYPE -------------------------------
df['rain'] = pd.to_numeric(df['rain'], errors='coerce')
df['station'] = pd.to_numeric(df['station'], errors='coerce').astype('Int64')
df['date'] = pd.to_datetime(df['date'], format='%d-%b-%y', errors='coerce')
# pandas maps 2-digit years 82..89 to 2082..2089 -> shift back one century
df.loc[df['date'].dt.year > 2000, 'date'] -= pd.DateOffset(years=100)
df['obstype'] = df['obstype'].str.strip()
df['network'] = np.where(df.sitegrid.str.endswith('RGP'), 'Princeton', 'LTER')
df['missing_flag'] = df.rain <= MISSING + 1e-6
df.loc[df.missing_flag, 'rain'] = np.nan
assert df.date.notna().all(), 'unparsed dates present'

# ----------------------------- 5. GEOREFERENCE -------------------------------
df['grid'] = df.sitegrid.str[:4]
geo = df.grid.str.fullmatch(r'\d{4}')          # 3 LTER sites use letter codes
df['SS'] = np.where(geo, df.grid.str[:2], np.nan)
df['EE'] = np.where(geo, df.grid.str[2:], np.nan)
df['SS'] = pd.to_numeric(df.SS); df['EE'] = pd.to_numeric(df.EE)
df['utm_n'] = GRID_N0 - df.SS * CELL
df['utm_e'] = GRID_E0 + df.EE * CELL
tr = pyproj.Transformer.from_crs('EPSG:32614', 'EPSG:4326', always_xy=True)
lon, lat = tr.transform(df.utm_e.values, df.utm_n.values)
df['lon'], df['lat'] = lon, lat

stations = (df.dropna(subset=['utm_n'])
              .groupby(['sitegrid', 'network'], as_index=False)
              .agg(station=('station', 'first'), utm_n=('utm_n', 'first'),
                   utm_e=('utm_e', 'first'), lat=('lat', 'first'), lon=('lon', 'first'),
                   n_rec=('rain', 'size'), n_valid=('rain', 'count'),
                   first=('date', 'min'), last=('date', 'max')))
print('\nGeoreferenced gauges: %d (LTER %d / Princeton %d) | ungeoreferenced (letter codes): %s'
      % (len(stations), (stations.network == 'LTER').sum(),
         (stations.network == 'Princeton').sum(),
         sorted(df.loc[~geo, 'sitegrid'].unique())))
print('Bounding box check  ->  lat %.3f..%.3f  lon %.3f..%.3f   (metadata: 39.07..39.11 / -96.61..-96.55)'
      % (stations.lat.min(), stations.lat.max(), stations.lon.min(), stations.lon.max()))
span_n = (stations.utm_n.max() - stations.utm_n.min()) / 1000
span_e = (stations.utm_e.max() - stations.utm_e.min()) / 1000
print('Network extent %.1f km (N-S) x %.1f km (E-W) = %.1f km2  ->  %.2f km2 per gauge'
      % (span_n, span_e, span_n * span_e, span_n * span_e / len(stations)))

# nearest-neighbour geometry
X = stations[['utm_e', 'utm_n']].values
D = np.sqrt(((X[:, None, :] - X[None, :, :]) ** 2).sum(-1))
np.fill_diagonal(D, np.inf)
print('Nearest-neighbour distance: min %.0f m | median %.0f m | max %.0f m'
      % (D.min(1).min(), np.median(D.min(1)), D.min(1).max()))

# ----------------------------- 6. PIVOT + DAILY SUMMARY ----------------------
piv = df.pivot_table(index='date', columns='sitegrid', values='rain', aggfunc='mean')
piv = piv.reindex(columns=sorted(piv.columns))

daily = pd.DataFrame({
    'n_gauges': piv.notna().sum(1),
    'areal_mean': piv.mean(1),
    'areal_sd': piv.std(1),
    'areal_min': piv.min(1),
    'areal_max': piv.max(1),
    'wet_fraction': (piv > 0.2).sum(1) / piv.notna().sum(1),   # gauges reporting rain
})
daily['cv'] = daily.areal_sd / daily.areal_mean.replace(0, np.nan)
daily['range_mm'] = daily.areal_max - daily.areal_min
daily['wet_day'] = daily.areal_mean > WET_MM
daily['year'] = daily.index.year

print('\n--- Data availability by year ---')
tab = daily.groupby('year').agg(days=('n_gauges', 'size'),
                                mean_gauges=('n_gauges', 'mean'),
                                max_gauges=('n_gauges', 'max'),
                                wet_days=('wet_day', 'sum'),
                                days_ge10=('n_gauges', lambda s: (s >= 10).sum()),
                                days_ge20=('n_gauges', lambda s: (s >= 20).sum())).round(1)
print(tab.to_string())

print('\n--- Usable sample for areal-average work ---')
for thr in [5, 8, 10, 15, 20, 25, 30]:
    sub = daily[daily.n_gauges >= thr]
    print('  >= %2d gauges : %3d days | %3d wet days' % (thr, len(sub), sub.wet_day.sum()))

W = daily[(daily.n_gauges >= 10) & daily.wet_day]
print('\nWet days with >=10 gauges: n=%d | spatial CV median %.2f (IQR %.2f-%.2f)'
      % (len(W), W.cv.median(), W.cv.quantile(.25), W.cv.quantile(.75)))
print('  max-min spread across the network: median %.1f mm | p90 %.1f mm | max %.1f mm'
      % (W.range_mm.median(), W.range_mm.quantile(.9), W.range_mm.max()))
part = W[(W.wet_fraction > 0) & (W.wet_fraction < 1)]
print('  partially-wetted days (some gauges dry while others report rain): %d / %d = %.0f%%'
      % (len(part), len(W), 100 * len(part) / len(W)))


# rebuild the coordinate lookup (shared by every spatial step below)
coord = {r.sitegrid: (r.utm_e, r.utm_n) for r in stations.itertuples()}
BLUE, RED, GREEN, GREY = '#1b6ca8', '#d1495b', '#2a9d8f', '#888888'

# =============================================================================
#  PART A  —  DATA-DESCRIPTION FIGURES  (context)
# =============================================================================
span_e = (stations.utm_e.max() - stations.utm_e.min()) / 1000
span_n = (stations.utm_n.max() - stations.utm_n.min()) / 1000

# F1 - network map
fig, ax = plt.subplots(figsize=(5.2, 5))
for net, mk, c in [('LTER', 's', '#1b6ca8'), ('Princeton', 'o', '#d1495b')]:
    s = stations[stations.network == net]
    ax.scatter(s.utm_e / 1000, s.utm_n / 1000, marker=mk, s=42, c=c, label='%s (n=%d)' % (net, len(s)),
               edgecolor='k', linewidth=.4, zorder=3)
ax.set_xlabel('UTM Easting Zone 14 [km]'); ax.set_ylabel('UTM Northing [km]')
ax.set_title('FIFE / Konza LTER daily rain-gauge network\n%.1f x %.1f km, %.2f km$^2$ per gauge'
             % (span_e, span_n, span_n * span_e / len(stations)))
ax.set_aspect('equal'); ax.legend(frameon=True)
fig.savefig(os.path.join(OUTDIR, 'F1_network_map.png')); plt.close(fig)
# F2 - availability matrix
fig, ax = plt.subplots(figsize=(9, 4.2))
M = piv.notna().astype(float).T
ax.imshow(M.values, aspect='auto', cmap='Greys', interpolation='nearest',
          extent=[0, M.shape[1], M.shape[0], 0])
ax.set_yticks(np.arange(M.shape[0]) + .5); ax.set_yticklabels(M.index, fontsize=5)
xt = np.linspace(0, M.shape[1] - 1, 10).astype(int)
ax.set_xticks(xt + .5); ax.set_xticklabels([piv.index[i].strftime('%Y-%m') for i in xt], rotation=45)
ax.set_title('Valid daily record availability (black = data present)'); ax.grid(False)
fig.savefig(os.path.join(OUTDIR, 'F2_availability.png')); plt.close(fig)
# F3 - concurrent gauges through time
fig, ax = plt.subplots(figsize=(8, 2.8))
ax.plot(daily.index, daily.n_gauges, lw=.8, color='#333')
ax.axhline(10, ls='--', c='#d1495b', lw=.8); ax.axhline(20, ls=':', c='#d1495b', lw=.8)
ax.set_ylabel('concurrent gauges'); ax.set_title('Number of gauges reporting a valid daily total')
fig.savefig(os.path.join(OUTDIR, 'F3_concurrent_gauges.png')); plt.close(fig)

# =============================================================================
#  PART B  —  DESIGN-CURVE ANALYSIS  (point-to-areal representativeness)
# =============================================================================
def _pip2(pkg):
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)
    if r.returncode:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                        '--break-system-packages', pkg], check=False)
try:
    import lightgbm as lgb
except ImportError:
    _pip2('lightgbm'); import lightgbm as lgb
try:
    import shap
except ImportError:
    _pip2('shap'); import shap
from scipy import stats
import itertools

SUMMARY = []
def note(s):
    print(s); SUMMARY.append(s)

Pgeo = piv[[c for c in piv.columns if c in coord]]
def wet_days(min_gauges):
    m = (daily.n_gauges >= min_gauges) & (daily.areal_mean > WET_MM)
    return Pgeo.loc[m]

# ---- B1. k-gauge accuracy curve (reference = all-gauge areal mean) ----
# --- Thiessen (area-weighted) reference machinery, for a robustness check ---
try:
    from shapely.geometry import Polygon, MultiPoint
except ImportError:
    _pip2('shapely'); from shapely.geometry import Polygon, MultiPoint
from scipy.spatial import Voronoi

_gXY = np.array([coord[c] for c in Pgeo.columns])
_domain = MultiPoint([tuple(p) for p in _gXY]).convex_hull.buffer(300)   # network footprint

def _voronoi_finite(vor):
    new_regions = []; new_vertices = vor.vertices.tolist()
    center = vor.points.mean(axis=0); radius = np.ptp(vor.points, axis=0).max() * 2
    ridges = {}
    for (p1, p2), (v1, v2) in zip(vor.ridge_points, vor.ridge_vertices):
        ridges.setdefault(p1, []).append((p2, v1, v2)); ridges.setdefault(p2, []).append((p1, v1, v2))
    for p1, reg in enumerate(vor.point_region):
        verts = vor.regions[reg]
        if all(v >= 0 for v in verts):
            new_regions.append(verts); continue
        nr = [v for v in verts if v >= 0]
        for p2, v1, v2 in ridges[p1]:
            if v2 < 0: v1, v2 = v2, v1
            if v1 >= 0: continue
            t = vor.points[p2] - vor.points[p1]; t /= np.linalg.norm(t); n = np.array([-t[1], t[0]])
            mid = vor.points[[p1, p2]].mean(axis=0)
            far = vor.vertices[v2] + np.sign(np.dot(mid - center, n)) * n * radius
            nr.append(len(new_vertices)); new_vertices.append(far.tolist())
        vs = np.asarray([new_vertices[v] for v in nr]); c = vs.mean(axis=0)
        nr = np.array(nr)[np.argsort(np.arctan2(vs[:, 1] - c[1], vs[:, 0] - c[0]))]
        new_regions.append(nr.tolist())
    return new_regions, np.asarray(new_vertices)

_wcache = {}
def thiessen_weights(idx):
    key = tuple(idx)
    if key in _wcache: return _wcache[key]
    if len(idx) < 4:
        w = np.full(len(idx), 1 / len(idx)); _wcache[key] = w; return w
    try:
        vor = Voronoi(_gXY[idx]); regs, verts = _voronoi_finite(vor)
        w = np.array([max(Polygon(verts[r]).intersection(_domain).area, 0.0) for r in regs])
        w = w / w.sum() if w.sum() > 0 else np.full(len(idx), 1 / len(idx))
    except Exception:
        w = np.full(len(idx), 1 / len(idx))
    _wcache[key] = w; return w

# --- k-curve with day-cluster bootstrap (resamples storms, the independent unit) ---
def k_curve(P, ks, ref='unweighted', m_sub=40, B=300, seed=0):
    rng = np.random.default_rng(seed); A = P.values; rows = []
    for k in ks:
        day_rel, day_abs, day_det, vdays = {}, {}, {}, []
        for i in range(len(P)):
            idx = np.where(~np.isnan(A[i]))[0]
            if len(idx) < k:
                continue
            v = A[i, idx]
            R = v.mean() if ref == 'unweighted' else float((v * thiessen_weights(idx)).sum())
            isw = R > WET_MM; er, ab, de = [], [], []
            for _ in range(m_sub):
                s = rng.choice(idx, k, replace=False); est = A[i, s].mean()
                er.append((est - R) / R); ab.append(est - R); de.append((est > WET_MM) == isw)
            day_rel[i] = np.array(er); day_abs[i] = np.array(ab)
            day_det[i] = np.array(de); vdays.append(i)
        vdays = np.array(vdays)
        allr = np.concatenate([day_rel[i] for i in vdays])
        alla = np.concatenate([day_abs[i] for i in vdays])
        alld = np.concatenate([day_det[i] for i in vdays])
        bt = []
        for _ in range(B):
            ds = rng.choice(vdays, len(vdays), replace=True)
            rr = np.concatenate([day_rel[i] for i in ds])
            bt.append(100 * np.sqrt((rr ** 2).mean()))
        rows.append((k, len(vdays), np.sqrt((alla ** 2).mean()), 100 * np.sqrt((allr ** 2).mean()),
                     np.percentile(bt, 2.5), np.percentile(bt, 97.5),
                     100 * (np.abs(allr) <= 0.20).mean(), 100 * (np.abs(allr) <= 0.10).mean(),
                     100 * alld.mean()))
    return pd.DataFrame(rows, columns=['k', 'n_days', 'RMSE_mm', 'relRMSE_pct',
                                       'relRMSE_lo', 'relRMSE_hi', 'within20_pct',
                                       'within10_pct', 'detect_pct'])

KC     = k_curve(wet_days(10), range(1, 9), ref='unweighted')
KC_thi = k_curve(wet_days(10), range(1, 9), ref='thiessen')
KChi   = k_curve(wet_days(20), [1, 2, 3, 5, 7, 10, 13, 16], ref='unweighted')
note('\n=== k-gauge accuracy curve (>=10-gauge wet days, day-cluster bootstrap) ===')
note(KC.round(1).to_string(index=False))
note('\n=== high-density extension (>=20-gauge wet days, 1987) ===')
note(KChi.round(1).to_string(index=False))

# --- reference robustness: Thiessen vs unweighted areal mean, per day (>=8 gauges) ---
_A = Pgeo.values; _rob = []
for i in range(len(Pgeo)):
    if not ((daily.n_gauges.iloc[i] >= 8) and (daily.areal_mean.iloc[i] > WET_MM)):
        continue
    idx = np.where(~np.isnan(_A[i]))[0]
    if len(idx) < 8:
        continue
    v = _A[i, idx]
    _rob.append((v.mean(), float((v * thiessen_weights(idx)).sum())))
ROB = pd.DataFrame(_rob, columns=['unweighted', 'thiessen'])
_corr = ROB.unweighted.corr(ROB.thiessen)
_reldiff = (ROB.thiessen - ROB.unweighted) / ROB.unweighted
note('\n=== reference robustness (Thiessen vs unweighted, n=%d, >=8 gauges) ===' % len(ROB))
note('  correlation %.4f | median rel. diff %+.1f%% | p90 |diff| %.1f%% | max |diff| %.1f%%'
     % (_corr, 100 * _reldiff.median(), 100 * _reldiff.abs().quantile(.9), 100 * _reldiff.abs().max()))
note('  design curve under the two references (relRMSE %%): k=1 %.1f vs %.1f | k=5 %.1f vs %.1f'
     % (KC.relRMSE_pct.iloc[0], KC_thi.relRMSE_pct.iloc[0],
        KC.relRMSE_pct.iloc[4], KC_thi.relRMSE_pct.iloc[4]))

# ---- B2. distribution-free tolerance radius delta_k (split conformal) ----
def conformal_tolerance(P, ks, alpha=0.10, draws=30):
    A = P.values; rows = []
    for k in ks:
        radii = []
        for dr in range(draws):
            rng = np.random.default_rng(dr); recs = []
            for i in range(len(P)):
                idx = np.where(~np.isnan(A[i]))[0]
                if len(idx) < k:
                    continue
                R = A[i, idx].mean(); s = rng.choice(idx, k, replace=False)
                recs.append((i, abs(A[i, s].mean() - R) / R))
            dd = pd.DataFrame(recs, columns=['day', 'rel'])
            days = dd.day.values.copy(); rng.shuffle(days)
            cal = dd[dd.day.isin(days[:int(.55 * len(days))])].rel.values
            radii.append(np.quantile(cal, np.ceil((len(cal) + 1) * (1 - alpha)) / len(cal)))
        rows.append((k, 100 * np.mean(radii), 100 * np.std(radii)))
    return pd.DataFrame(rows, columns=['k', 'tol_pct', 'tol_sd'])

TOL = conformal_tolerance(wet_days(10), [1, 2, 3, 4, 5, 6, 8])
note('\n=== distribution-free 90% tolerance radius by k ===')
note(TOL.round(1).to_string(index=False))
for tgt in [50, 30, 25, 20]:
    ok = TOL.loc[TOL.tol_pct <= tgt, 'k']
    note('  min gauges for +-%d%% at 90%% confidence: %s'
         % (tgt, int(ok.iloc[0]) if len(ok) else '>8'))

# ---- B3. variance-mean scaling (Taylor) + SHAP drivers ----
VAR_MIN = 8
geo_cnt = Pgeo.notna().sum(1)
sel = (geo_cnt >= VAR_MIN) & (daily.areal_mean > WET_MM)
Psel = Pgeo.loc[sel]; A = Psel.values; feat = []
for i in range(len(A)):
    idx = np.where(~np.isnan(A[i]))[0]
    v = A[i, idx]; doy = Psel.index[i].dayofyear
    if v.mean() <= WET_MM or v.std(ddof=1) == 0:
        continue
    feat.append((v.mean(), v.std(ddof=1), v.std(ddof=1) / v.mean(),
                 (v > 0.2).mean(), len(idx),
                 np.sin(2 * np.pi * doy / 365), np.cos(2 * np.pi * doy / 365)))
F = pd.DataFrame(feat, columns=['mean', 'sd', 'cv', 'wetfrac', 'n', 'sin', 'cos'])
b, a_ = np.polyfit(np.log(F['mean']), np.log(F.sd), 1)
r2 = np.corrcoef(np.log(F['mean']), np.log(F.sd))[0, 1] ** 2
note('\n=== Taylor power law:  SD = %.2f * mean^%.2f   (log-log R2 = %.2f, n=%d, >=%d gauges) ==='
     % (np.exp(a_), b, r2, len(F), VAR_MIN))
note('  exponent b = %.2f  ->  CV ~ mean^%.2f  (b<1: variability falls with storm size)' % (b, b - 1))

Xcols = ['mean', 'wetfrac', 'n', 'sin', 'cos']
X = F[Xcols].copy(); X['mean'] = np.log10(X['mean']); X = X.rename(columns={'mean': 'log10_mean'})
y = np.log10(F.sd)
mdl = lgb.LGBMRegressor(n_estimators=400, num_leaves=15, learning_rate=0.03,
                        min_child_samples=12, subsample=0.8, verbose=-1).fit(X, y)
note('  LightGBM fit to log10(spatial SD): in-sample R2 = %.2f' % mdl.score(X, y))
SV = shap.TreeExplainer(mdl).shap_values(X)
imp = pd.Series(np.abs(SV).mean(0), index=X.columns).sort_values(ascending=False)
note('  SHAP mean|value| ranking (drivers of spatial variability):')
for nm, val in imp.items():
    note('     %-12s %.3f' % (nm, val))

# ---- B4. correlation vs error (network-design caution) ----
pv = Pgeo.loc[daily.n_gauges >= 10]
pairs = []
for g1, g2 in itertools.combinations(pv.columns, 2):
    s = pv[[g1, g2]].dropna(); s = s[(s[g1] > 0.2) | (s[g2] > 0.2)]
    if len(s) >= 25:
        d = float(np.hypot(coord[g1][0] - coord[g2][0], coord[g1][1] - coord[g2][1]))
        nr = np.sqrt(((s[g1] - s[g2]) ** 2).mean()) / s.values.mean()
        agree = ((s[g1] > 0.2) == (s[g2] > 0.2)).mean()
        pairs.append((d, s[g1].corr(s[g2]), nr, agree))
PP = pd.DataFrame(pairs, columns=['dist_m', 'r', 'nrmse', 'wetdry_agree'])
PP['bin'] = pd.cut(PP.dist_m, [0, 500, 1000, 1500, 2000, 3000, 4000, 6000])
binned = PP.groupby('bin', observed=True).agg(d=('dist_m', 'mean'), r=('r', 'median'),
                                              nrmse=('nrmse', 'median')).reset_index(drop=True)

# =============================================================================
#  CORE-RESULT FIGURES
# =============================================================================
# F4 - design curve (centerpiece)
fig, ax = plt.subplots(1, 3, figsize=(13.5, 4.2), constrained_layout=True)
ax[0].fill_between(KC.k, KC.relRMSE_lo, KC.relRMSE_hi, color=BLUE, alpha=.18, label='95% day-cluster CI')
ax[0].plot(KC.k, KC.relRMSE_pct, 'o-', color=BLUE, label='relative RMSE (≥10-gauge days)')
ax[0].plot(TOL.k, TOL.tol_pct, 's--', color=RED, label='90% distribution-free tolerance')
ax[0].plot(KChi.k, KChi.relRMSE_pct, '^:', color=GREY, ms=5, label='≥20-gauge days (1987)')
for tgt in [20, 30]:
    ax[0].axhline(tgt, color='k', lw=.5, ls=':')
ax[0].set_xlabel('number of gauges k'); ax[0].set_ylabel('areal-mean error [%]')
ax[0].set_title('Point-to-areal accuracy'); ax[0].legend(fontsize=7.2, loc='upper right'); ax[0].set_ylim(0, None)
ax[1].plot(KC.k, KC.within20_pct, 'o-', color=BLUE, label='within ±20%')
ax[1].plot(KC.k, KC.within10_pct, 's-', color=GREEN, label='within ±10%')
ax[1].set_xlabel('number of gauges k'); ax[1].set_ylabel('probability [%]')
ax[1].set_title('Chance a k-gauge mean hits tolerance'); ax[1].legend(fontsize=8); ax[1].set_ylim(0, 101)
ax[2].plot(KC.k, KC.detect_pct, 'o-', color=RED)
ax[2].set_xlabel('number of gauges k'); ax[2].set_ylabel('wet/dry agreement [%]')
ax[2].set_title('Rain-occurrence detection'); ax[2].set_ylim(85, 100.5)
fig.suptitle('Gauge-density design curve for a ~22 km² area  (FIFE / Konza, %d wet days)' % len(wet_days(10)),
             fontweight='bold', fontsize=11)
fig.savefig(os.path.join(OUTDIR, 'F4_design_curve.png'), dpi=300); plt.close(fig)

# F5 - variance-mean scaling
fig, ax = plt.subplots(1, 2, figsize=(9.5, 4.0), constrained_layout=True)
xx = np.logspace(np.log10(F['mean'].min()), np.log10(F['mean'].max()), 50)
sc = ax[0].scatter(F['mean'], F.sd, c=F.wetfrac, cmap='viridis', s=26, edgecolor='k', linewidth=.2)
ax[0].plot(xx, np.exp(a_) * xx ** b, color=RED, lw=1.6,
           label='SD = %.2f·mean$^{%.2f}$\n$R^2$=%.2f' % (np.exp(a_), b, r2))
ax[0].set_xscale('log'); ax[0].set_yscale('log')
ax[0].set_xlabel('areal-mean rainfall [mm]'); ax[0].set_ylabel('spatial SD across gauges [mm]')
ax[0].set_title("Variance–mean scaling (Taylor's law)"); ax[0].legend(fontsize=8)
fig.colorbar(sc, ax=ax[0], label='fraction of gauges wet', fraction=.046, pad=.02)
ax[1].scatter(F['mean'], F.cv, c=F.wetfrac, cmap='viridis', s=26, edgecolor='k', linewidth=.2)
ax[1].plot(xx, np.exp(a_) * xx ** (b - 1), color=RED, lw=1.6, label='CV ~ mean$^{%.2f}$' % (b - 1))
ax[1].set_xscale('log'); ax[1].set_xlabel('areal-mean rainfall [mm]'); ax[1].set_ylabel('spatial CV [-]')
ax[1].set_title('Variability falls with storm size'); ax[1].legend(fontsize=8)
fig.savefig(os.path.join(OUTDIR, 'F5_variance_mean_scaling.png'), dpi=300); plt.close(fig)

# F6 - SHAP drivers
fig = plt.figure(figsize=(7.5, 4.2))
shap.summary_plot(SV, X, show=False, plot_size=None)
plt.title('Drivers of daily spatial variability (SHAP)', fontsize=10)
plt.tight_layout(); plt.savefig(os.path.join(OUTDIR, 'F6_shap_drivers.png'), dpi=300); plt.close(fig)

# F7 - correlation vs error  (legend placed BELOW the x-axis label with reserved space)
fig, ax1 = plt.subplots(figsize=(7.0, 5.0))
ax1.scatter(PP.dist_m, PP.r, s=10, alpha=.30, color=BLUE)
l1, = ax1.plot(binned.d, binned.r, 'o-', color=BLUE, label='Pearson r (left)')
ax1.set_xlabel('gauge separation [m]', labelpad=6); ax1.set_ylabel('Pearson r', color=BLUE)
ax1.tick_params(axis='y', labelcolor=BLUE); ax1.set_ylim(.85, 1.0)
ax2 = ax1.twinx()
ax2.scatter(PP.dist_m, PP.nrmse, s=10, alpha=.30, color=RED)
l2, = ax2.plot(binned.d, binned.nrmse, 's--', color=RED, label='pairwise NRMSE (right)')
ax2.set_ylabel('pairwise NRMSE [-]', color=RED); ax2.tick_params(axis='y', labelcolor=RED)
ax1.set_title('Correlation stays high while error grows\n(a caution for correlation-based network design)', fontsize=10)
fig.subplots_adjust(bottom=0.22, top=0.88)                    # reserve room so legend clears the x-label
fig.legend([l1, l2], [t.get_label() for t in (l1, l2)],
           loc='lower center', bbox_to_anchor=(0.5, 0.01), ncol=2, fontsize=8.5, frameon=True)
fig.savefig(os.path.join(OUTDIR, 'F7_correlation_vs_error.png'), dpi=300); plt.close(fig)

# F8 - the three largest network-wide wet days, as gauge maps
top = W.sort_values('areal_mean', ascending=False).head(3).index
fig, axes = plt.subplots(1, 3, figsize=(13.5, 5.0), constrained_layout=True)
for ax, d in zip(axes, top):
    v = piv.loc[d].dropna()
    v = v[[i in coord for i in v.index]]
    x = np.array([coord[i][0] / 1000 for i in v.index])
    y = np.array([coord[i][1] / 1000 for i in v.index])
    sc = ax.scatter(x, y, c=v.values, s=150, cmap='YlGnBu', edgecolor='k',
                    linewidth=.6, zorder=3)
    cb = fig.colorbar(sc, ax=ax, fraction=0.047, pad=0.02)
    cb.set_label('daily rainfall [mm]', fontsize=8.5)
    cb.ax.tick_params(labelsize=7.5)
    # two-line title: bold date + descriptive stats underneath, with breathing room
    ax.set_title('%s   (%d gauges)\nareal mean %.0f mm  ·  range %.0f–%.0f mm  ·  CV %.2f'
                 % (d.strftime('%-d %b %Y'), len(v), v.mean(), v.min(), v.max(),
                    v.std() / v.mean()), fontsize=9.5, pad=10)
    ax.set_xlabel('UTM Easting [km]', fontsize=8.5)
    ax.set_aspect('equal')
    ax.margins(0.18)
    ax.tick_params(labelsize=7.5)
axes[0].set_ylabel('UTM Northing [km]', fontsize=8.5)
fig.suptitle('Spatial pattern of the three heaviest network-wide wet days over the ~22 km$^2$ FIFE network',
             fontsize=11, fontweight='bold')
fig.savefig(os.path.join(OUTDIR, 'F8_largest_events.png'), dpi=300)
plt.close(fig)

# F9 - within-network variability detail (CV and absolute spread vs areal mean)
fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.8), constrained_layout=True)
axes[0].scatter(W.areal_mean, W.cv, s=18, c=W.n_gauges, cmap='viridis', edgecolor='k', linewidth=.2)
axes[0].set_xscale('log'); axes[0].set_xlabel('areal-mean rainfall [mm]'); axes[0].set_ylabel('spatial CV [-]')
axes[0].set_title('Within-network variability, wet days (n=%d)' % len(W))
sc = axes[1].scatter(W.areal_mean, W.range_mm, s=18, c=W.n_gauges, cmap='viridis', edgecolor='k', linewidth=.2)
axes[1].plot([.5, W.areal_mean.max()], [.5, W.areal_mean.max()], 'k--', lw=.7, label='1:1')
axes[1].set_xscale('log'); axes[1].set_yscale('log')
axes[1].set_xlabel('areal-mean rainfall [mm]'); axes[1].set_ylabel('max − min across gauges [mm]')
axes[1].set_title('Absolute spread within ~22 km$^2$'); axes[1].legend(fontsize=8)
fig.colorbar(sc, ax=axes[1], label='gauges', fraction=.046, pad=.02)
fig.savefig(os.path.join(OUTDIR, 'F9_variability_detail.png'), dpi=300); plt.close(fig)

# F10 - pairwise agreement vs separation (three complementary metrics)
fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.6), constrained_layout=True)
for ax, col, ttl in zip(axes, ['r', 'nrmse', 'wetdry_agree'],
                        ['Pearson r (rainy days)', 'pairwise NRMSE [-]', 'wet/dry agreement [-]']):
    ax.scatter(PP.dist_m, PP[col], s=12, alpha=.4, edgecolor='none', color=BLUE)
    bb = PP.groupby('bin', observed=True)[col].median()
    ax.plot([iv.mid for iv in bb.index], bb.values, 'o-', color=RED, ms=4, lw=1.2)
    ax.set_xlabel('gauge separation [m]'); ax.set_title(ttl)
fig.suptitle('Inter-gauge agreement as a function of separation', fontweight='bold', fontsize=10.5)
fig.savefig(os.path.join(OUTDIR, 'F10_distance_agreement.png'), dpi=300); plt.close(fig)

# F11 - reference robustness (Thiessen area-weighted vs unweighted)
fig, ax = plt.subplots(1, 2, figsize=(9.5, 4.0), constrained_layout=True)
mx = max(ROB.unweighted.max(), ROB.thiessen.max())
ax[0].scatter(ROB.unweighted, ROB.thiessen, s=20, color=BLUE, edgecolor='k', linewidth=.2, alpha=.8)
ax[0].plot([0, mx], [0, mx], 'k--', lw=.7, label='1:1')
ax[0].set_xlabel('unweighted areal mean [mm]'); ax[0].set_ylabel('Thiessen areal mean [mm]')
ax[0].set_title('Reference agreement (r = %.4f)' % _corr); ax[0].legend(fontsize=8)
ax[1].fill_between(KC.k, KC.relRMSE_lo, KC.relRMSE_hi, color=BLUE, alpha=.15)
ax[1].plot(KC.k, KC.relRMSE_pct, 'o-', color=BLUE, label='unweighted reference')
ax[1].plot(KC_thi.k, KC_thi.relRMSE_pct, 's--', color=RED, label='Thiessen reference')
ax[1].set_xlabel('number of gauges k'); ax[1].set_ylabel('areal-mean error [%]')
ax[1].set_title('Design curve is robust to reference choice'); ax[1].legend(fontsize=8); ax[1].set_ylim(0, None)
fig.suptitle('Robustness of the areal reference (area-weighted vs unweighted)', fontweight='bold', fontsize=10.5)
fig.savefig(os.path.join(OUTDIR, 'F11_reference_robustness.png'), dpi=300); plt.close(fig)

# =============================================================================
#  EXPORT
# =============================================================================
# ----------------------------- 9. EXPORT -------------------------------------
df.drop(columns=['grid']).to_csv(os.path.join(OUTDIR, 'fife_rain_long.csv'), index=False)
piv.to_csv(os.path.join(OUTDIR, 'fife_rain_pivot.csv'))
stations.to_csv(os.path.join(OUTDIR, 'fife_stations.csv'), index=False)
daily.to_csv(os.path.join(OUTDIR, 'fife_daily_summary.csv'))
PP.drop(columns=['bin']).to_csv(os.path.join(OUTDIR, 'fife_pairwise.csv'), index=False)
tab.to_csv(os.path.join(OUTDIR, 'fife_year_table.csv'))
KC.to_csv(os.path.join(OUTDIR, 'D_k_curve.csv'), index=False)
KC_thi.to_csv(os.path.join(OUTDIR, 'D_k_curve_thiessen.csv'), index=False)
ROB.to_csv(os.path.join(OUTDIR, 'D_reference_robustness.csv'), index=False)
KChi.to_csv(os.path.join(OUTDIR, 'D_k_curve_dense.csv'), index=False)
TOL.to_csv(os.path.join(OUTDIR, 'D_tolerance_curve.csv'), index=False)
F.to_csv(os.path.join(OUTDIR, 'D_daily_variability.csv'), index=False)
binned.to_csv(os.path.join(OUTDIR, 'D_corr_vs_error.csv'), index=False)
with open(os.path.join(OUTDIR, 'D_results_summary.txt'), 'w') as fh:
    fh.write('FIFE point-to-areal representativeness — key results\n')
    fh.write('doi:10.3334/ORNLDAAC/29 | network ~22 km2, %d georeferenced gauges\n\n' % len(stations))
    fh.write('\n'.join(SUMMARY))

import shutil
_zip = shutil.make_archive('fife_full_outputs', 'zip', OUTDIR)
print('\nBundled -> %s (%.0f KB) | %d files' % (os.path.basename(_zip),
      os.path.getsize(_zip) / 1024, len(os.listdir(OUTDIR))))
if IN_COLAB:
    from google.colab import files as _f
    _f.download('fife_full_outputs.zip')


>> Pick your .rar file (e.g. fife_sur_met_rain_day_29_1-...rar)


Saving fife_sur_met_rain_day_29_1-20260717_175014.rar to fife_sur_met_rain_day_29_1-20260717_175014.rar
Using archive: fife_sur_met_rain_day_29_1-20260717_175014.rar (71562 bytes)
Extracted 89 granule files -> fife_work/
Parsed 10034 observation records from 89 granules

Georeferenced gauges: 39 (LTER 9 / Princeton 30) | ungeoreferenced (letter codes): ['K1BF-RGL', 'K4BF-RGL', 'KUBF-RGL']
Bounding box check  ->  lat 39.065..39.107  lon -96.610..-96.554   (metadata: 39.07..39.11 / -96.61..-96.55)
Network extent 4.6 km (N-S) x 4.8 km (E-W) = 22.1 km2  ->  0.57 km2 per gauge
Nearest-neighbour distance: min 200 m | median 447 m | max 1720 m

--- Data availability by year ---
      days  mean_gauges  max_gauges  wet_days  days_ge10  days_ge20
year                                                               
1982    87          1.2           2        87          0          0
1983    93          2.7           4        80          0          0
1984    96          4.0           4        72   

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>